# Importing and vizualizing dataset

In [1]:
import torch
import torch.nn as nn
import torchvision
import pandas as pd
import torchvision.transforms as transforms

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

from pathlib import Path

In [2]:
filedir = Path('./datasets/')

train_dataframe = pd.read_csv(filedir / 'train_data.csv')
test_dataframe = pd.read_csv(filedir / 'test_data.csv')

In [3]:
train_dataframe.head()

,ID,patient_id,breast density,left or right breast,image view,abnormality id,abnormality type,calc type,calc distribution,assessment,pathology,subtlety
0,Calc-Training_P_00005_RIGHT_CC_1,P_00005,3,RIGHT,CC,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
1,Calc-Training_P_00005_RIGHT_MLO_1,P_00005,3,RIGHT,MLO,1,calcification,AMORPHOUS,CLUSTERED,3,MALIGNANT,3
2,Calc-Training_P_00007_LEFT_CC_1,P_00007,4,LEFT,CC,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
3,Calc-Training_P_00007_LEFT_MLO_1,P_00007,4,LEFT,MLO,1,calcification,PLEOMORPHIC,LINEAR,4,BENIGN,4
4,Calc-Training_P_00008_LEFT_CC_1,P_00008,1,LEFT,CC,1,calcification,NaN,REGIONAL,2,BENIGN_WITHOUT_CALLBACK,3


# Data preprocessing

In [4]:
class BreastCancerDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image_path = f"{self.image_dir}/{row['ID']}.jpg"  # Ensure path matches your structure
        image = Image.open(image_path).convert("L")  # Grayscale image
        
        label = row['pathology']  # Target label
        label = 0 if label == 'BENIGN_WITHOUT_CALLBACK' else 1 if label == 'BENIGN' else 2  # Encoding classes

        additional_features = row[['abnormality id', 'calc distribution']].astype(float)  # Additional features to consider
        additional_features = torch.tensor(additional_features.values, dtype=torch.float32)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label, additional_features

In [5]:
transform = transforms.Compose([
    transforms.Resize((64, 64)),  # Resize every image to 64 x 64 matrix
    transforms.ToTensor()  # Default normalization to [0, 1]
])

In [6]:
label_encoder = LabelEncoder()

train_dataframe['calc distribution'] = label_encoder.fit_transform(train_dataframe['calc distribution'])
test_dataframe['calc distribution'] = label_encoder.transform(test_dataframe['calc distribution'])

train_dataset = BreastCancerDataset(train_dataframe, filedir / 'train_cropped_images/', transform=transform)
test_dataset = BreastCancerDataset(test_dataframe, filedir / 'test_cropped_images/', transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True)

# Definition and neural network model design

In [28]:
class CNN(nn.Module):
    def __init__(self, num_classes, additional_features_size=0):
        super(CNN, self).__init__()
        # Convolutional layers
        self.conv_layer1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.batch_norm1 = nn.BatchNorm2d(32)
        self.conv_layer2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.batch_norm2 = nn.BatchNorm2d(64)
        self.max_pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv_layer3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.batch_norm3 = nn.BatchNorm2d(64)
        self.conv_layer4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.batch_norm4 = nn.BatchNorm2d(128)
        self.max_pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.dropout = nn.Dropout(0.3)  # Dropout for regularization

        # Fully connected layers
        self.flattened_size = None  # To be calculated dynamically
        self.fc1 = None  # To be initialized dynamically
        self.relu = nn.ReLU()
        
        # Additional features
        self.additional_features_size = additional_features_size
        if self.additional_features_size > 0:
            self.fc_additional = nn.Linear(self.additional_features_size, 64)
        
        # Final classification layer
        self.fc_final = nn.Linear(128 if additional_features_size > 0 else 64, num_classes)
        
    def forward(self, x, additional_features=None):
        # Convolutional layers with ReLU activation
        x = self.relu(self.conv_layer1(x))
        x = self.relu(self.batch_norm1(x))
        x = self.relu(self.conv_layer2(x))
        x = self.relu(self.batch_norm2(x))
        x = self.max_pool1(x)
        
        x = self.relu(self.conv_layer3(x))
        x = self.relu(self.batch_norm3(x))
        x = self.relu(self.conv_layer4(x))
        x = self.relu(self.batch_norm4(x))
        x = self.max_pool2(x)

        x = self.dropout(x)
        # Flatten dynamically
        x = x.view(x.size(0), -1)
        
        # Initialize fc1 if not done (dynamic size)
        if self.fc1 is None:
            self.flattened_size = x.size(1)
            self.fc1 = nn.Linear(self.flattened_size, 64)
            self.fc1.to(x.device)
        
        x = self.relu(self.fc1(x))
        
        # Add additional features if provided
        if additional_features is not None and self.additional_features_size > 0:
            additional_features = self.relu(self.fc_additional(additional_features))
            x = torch.cat((x, additional_features), dim=1)
        
        # Final classification layer
        x = self.fc_final(x)
        return x


## Hyperparameter definition

In [29]:
num_classes = len(train_dataframe['pathology'].unique())  # There are 3 target classes

num_epochs = 100  # Training period

model = CNN(num_classes)  # Set model parameters

optimizer = torch.optim.Adam(model.parameters(), weight_decay=1e-4)  # Default learning rate 10^-3 sounds fine

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)  # Change learning rate if stagnate

criterion = nn.CrossEntropyLoss()  # Cross entropy loss between input logits and target

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Whether to train on GPU (cuda) or CPU

model.to(device)

CNN(
  (conv_layer1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv_layer2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (max_pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv_layer3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm3): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv_layer4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (batch_norm4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (max_pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.3, inplace=False)
  (relu): ReLU()
  (fc_final): Linear(in_features=6

### Training loop

In [30]:
for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    all_preds = []
    all_labels = []

    # Training loop
    for images, labels, additional_features in train_loader:
        images = images.to(device)
        labels = labels.to(device)
        additional_features = additional_features.to(device)
        
        # Forward pass
        outputs = model(images, additional_features)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Accumulate loss and predictions
        train_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    train_loss /= len(train_loader)
    train_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}, Accuracy: {train_acc:.4f}", end=", ")
    
    # Validation loop
    model.eval()
    val_loss = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels, additional_features in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            additional_features = additional_features.to(device)
            
            outputs = model(images, additional_features)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    val_loss /= len(test_loader)
    val_acc = accuracy_score(all_labels, all_preds)
    scheduler.step(val_loss)
    print(f"Validation Loss: {val_loss:.4f}, Accuracy: {val_acc:.4f}")


Epoch 1/100, Loss: 0.9844, Accuracy: 0.4585, Validation Loss: 1.0242, Accuracy: 0.4724
Epoch 2/100, Loss: 0.8762, Accuracy: 0.5447, Validation Loss: 0.9032, Accuracy: 0.5215
Epoch 3/100, Loss: 0.8600, Accuracy: 0.5628, Validation Loss: 0.8837, Accuracy: 0.5491
Epoch 4/100, Loss: 0.8223, Accuracy: 0.5810, Validation Loss: 0.8748, Accuracy: 0.4969
Epoch 5/100, Loss: 0.7922, Accuracy: 0.6088, Validation Loss: 0.8845, Accuracy: 0.5491
Epoch 6/100, Loss: 0.7685, Accuracy: 0.6315, Validation Loss: 0.9100, Accuracy: 0.5337
Epoch 7/100, Loss: 0.7399, Accuracy: 0.6464, Validation Loss: 1.0126, Accuracy: 0.4969
Epoch 8/100, Loss: 0.7216, Accuracy: 0.6710, Validation Loss: 0.8812, Accuracy: 0.5092
Epoch 9/100, Loss: 0.6861, Accuracy: 0.6859, Validation Loss: 0.8839, Accuracy: 0.5031
Epoch 10/100, Loss: 0.6411, Accuracy: 0.7144, Validation Loss: 0.9444, Accuracy: 0.5092
Epoch 11/100, Loss: 0.5538, Accuracy: 0.7811, Validation Loss: 0.9405, Accuracy: 0.5031
Epoch 12/100, Loss: 0.5179, Accuracy: 0.8

In [32]:
save_path = Path("./model")
torch.save(model.state_dict(), save_path)